- **Name:** 20.3_streaming_bad_data
- **Author:** Shamas Imran
- **Desciption:** Handling bad or corrupt data in streaming pipelines
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Simulated bad records in streaming input  
                                              handled schema mismatch  
-->

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 3, Finished, Available, Finished)

In [2]:
# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("CSV_Streaming_Error_Handling")
        .getOrCreate()
)

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 4, Finished, Available, Finished)

In [3]:
# ------------------------------
# 1) Paths (your folders)
# ------------------------------
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath = masterPath + "csv_input"
checkpointPath = masterPath + "checkpoints/csv_query"
outputPath = masterPath + "csv_output"
badRecordsPath = masterPath + "baddata_output"


StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 5, Finished, Available, Finished)

In [4]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 6, Finished, Available, Finished)

In [5]:
# Create directories
import os

os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)
os.makedirs(badRecordsPath, exist_ok=True)

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 7, Finished, Available, Finished)

In [6]:
import pandas as pd
import os

lakehouse_folder = inputPath 

valid_data = [
    {"id": 1, "name": "Ali", "score": 85, "event_time": "2025-08-18 10:00:00"},
    {"id": 2, "name": "Sara", "score": 90, "event_time": "2025-08-18 10:05:00"},
    {"id": 3, "name": "Omar", "score": 75, "event_time": "2025-08-18 10:10:00"}
]

valid_df = pd.DataFrame(valid_data)
valid_path = os.path.join(lakehouse_folder, "Valid_data.csv")
valid_df.to_csv(valid_path, index=False, header=True)

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 8, Finished, Available, Finished)

In [7]:
# ------------------------------------------------------------
# 3) Define Schema (expected columns)
# ------------------------------------------------------------
# NOTE: we add `_corrupt_record` column to capture invalid rows.
schema = StructType([
    StructField("id", IntegerType(), True),         # should be integer
    StructField("name", StringType(), True),        # string field
    StructField("score", IntegerType(), True),      # should be integer
    StructField("event_time", TimestampType(), True), # event time (for future)
    StructField("_corrupt_record", StringType(), True)  # keeps the bad row
])

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 9, Finished, Available, Finished)

In [8]:
# ------------------------------------------------------------
# 4) Create Streaming DataFrame with Error Handling
# ------------------------------------------------------------
# mode = PERMISSIVE => put invalid rows into _corrupt_record
df_stream = (
    spark.readStream
         .option("header", "true")                           # CSV has header
         .schema(schema)                                     # enforce schema
         .option("mode", "PERMISSIVE")                       # keep bad rows
         .option("columnNameOfCorruptRecord", "_corrupt_record")
        #  .option("enforceSchema", "false")
         .csv(inputPath)                                     # folder to watch
)

# option("mode" ============>
# PERMISSIVE (default) → Keeps all rows; bad rows go into _corrupt_record column.
# DROPMALFORMED → Skips bad rows completely (they are dropped).
# FAILFAST → Stops the job immediately when a bad row is found.

# ------------------------------------------------------------
# 5) Separate Good vs Bad Records
# ------------------------------------------------------------
valid_rows = df_stream.filter(df_stream["_corrupt_record"].isNull())
bad_rows   = df_stream.filter(df_stream["_corrupt_record"].isNotNull())

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 10, Finished, Available, Finished)

In [9]:
# ------------------------------------------------------------
# 6) Write Valid Records to Output Folder
# ------------------------------------------------------------
valid_query = (
    valid_rows.writeStream
              .format("csv")                                 # write CSVs
              .option("path", outputPath)                    # output folder
              .option("header", "true") 
              .option("checkpointLocation", checkpointPath + "/valid") # unique checkpoint
              .outputMode("append")                          # append = new rows only
              .start()
)

# ------------------------------------------------------------
# 7) Write Bad Records to Quarantine Folder
# ------------------------------------------------------------
bad_query = (
    bad_rows.writeStream
            .format("csv")                                   # store bad rows separately
            .option("path", badRecordsPath)
            .option("header", "true") 
            .option("checkpointLocation", checkpointPath + "/bad") # unique checkpoint
            .outputMode("append")
            .start()
)

# ------------------------------------------------------------
# 8) Wait for Streams to Finish
# ------------------------------------------------------------
# This keeps your job running until you stop it manually
spark.streams.awaitAnyTermination()

StatementMeta(, 727959f7-bab9-4913-9228-819d159739ba, 11, Finished, Cancelled, Cancelled)

In [ ]:
"""
Valid_data.csv
id,name,score,event_time
1,Ali,85,2025-08-18 10:00:00
2,Sara,90,2025-08-18 10:05:00
3,Omar,75,2025-08-18 10:10:00

Valid2_data.csv
id,name,score,event_time
101,Akbar,70,2025-08-18 10:00:00
102,Aslam,93,2025-08-18 10:05:00
103,Mahim,87,2025-08-18 10:10:00

invalidd_data.csv
id,name,score,event_time,extra
501,Adeel,95,2025-08-18 10:20:00,                -- ⚠️ Extra column (ignored if enforceSchema=false)
502,Nadia,85,2025-08-18 10:25:00,extra_col_data  -- ⚠️ Extra column (ignored if enforceSchema=false)
503,Ahmed,75,2025-08-18 10:30:00                 -- ✅ Valid row (matches schema)
504,Sadia,NotANumber,2025-08-18 10:35:00,Extra   -- ❌ Invalid (score not integer)
505,Omar,88,                                     -- ⚠️ Missing event_time
506,Sarah,,2025-08-18 10:45:00,ExtraColumn       -- ⚠️ Missing score
507,Ali Khan,92,2025-08-18 10:50:00              -- ✅ Valid row
508,,81,2025-08-18 10:55:00,ExtraData            -- ⚠️ Missing name
509,Imran,90,invalid_timestamp,ExtraStuff        -- ❌ Malformed timestamp
510,Zara,78,2025-08-18 11:00:00                 -- ✅ Valid row
"""

In [3]:
df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load("Files/spark-streaming/csv_output/*.csv")
)
df.show()


df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load("Files/spark-streaming/baddata_output/*.csv")
)
df.show()


StatementMeta(, c3fcd7f2-adc4-4e3c-8c1b-176a6e44d82f, 5, Finished, Available, Finished)

+---+--------+---+------------------------+----+
|503|   Ahmed| 75|2025-08-18T10:30:00.000Z| _c4|
+---+--------+---+------------------------+----+
|505|    Omar| 88|                    NULL|NULL|
|507|Ali Khan| 92|    2025-08-18T10:50:...|NULL|
|510|    Zara| 78|    2025-08-18T11:00:...|NULL|
|102|   Aslam| 93|    2025-08-18T10:05:...|NULL|
|103|   Mahim| 87|    2025-08-18T10:10:...|NULL|
|  2|    Sara| 90|    2025-08-18T10:05:...|NULL|
|  3|    Omar| 75|    2025-08-18T10:10:...|NULL|
+---+--------+---+------------------------+----+

+---+-----+----+------------------------+---------------------------------+
|501|Adeel|  95|2025-08-18T10:20:00.000Z|501,Adeel,95,2025-08-18 10:20:00,|
+---+-----+----+------------------------+---------------------------------+
|502|Nadia|  85|    2025-08-18T10:25:...|             502,Nadia,85,2025...|
|504|Sadia|NULL|    2025-08-18T10:35:...|             504,Sadia,NotANum...|
|506|Sarah|NULL|    2025-08-18T10:45:...|             506,Sarah,,2025-0...|
|508